# RootBot: навчання моделі фігурок

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. На кнопці **Choose Files** вибери zip теки `data/datasets/test_claude_labeled_60frames` (у файловому менеджері: права кнопка → Compress).
4. Чекай ~30-60 хв, вкладку не закривай. В кінці завантажиться `rootbot_model.zip`.

Кадри: train 45, val 12, test 3 (окреме відео `VID_20261105_201500`).

In [ ]:
!pip install -q -U ultralytics

In [ ]:
import shutil, zipfile, re
from pathlib import Path
from google.colab import files
import torch
assert torch.cuda.is_available(), 'Немає GPU: Runtime -> Change runtime type -> T4 GPU'

SRC, DATA = Path('/content/src'), Path('/content/rootbot')
for name in files.upload():
    zipfile.ZipFile(name).extractall(SRC)

# Test: the separate video, never used in training. Val: whole blocks of the main video,
# because neighbouring frames are almost identical and would make the score look too good.
VAL = {64, 66, 68, 71, 74, 77, 80, 83, 140, 143, 146, 147}
def part(stem):
    if '_201500_' in stem:
        return 'test'
    return 'val' if int(stem[-4:]) in VAL else 'train'

counts = {'train': 0, 'val': 0, 'test': 0}
IMAGES = next(SRC.rglob('images'))
for img in sorted(IMAGES.glob('*.jpg')):
    p = part(img.stem)
    counts[p] += 1
    for kind, src in (('images', img), ('labels', IMAGES.parent / 'outlines' / f'{img.stem}.txt')):
        (DATA / kind / p).mkdir(parents=True, exist_ok=True)
        shutil.copy(src, DATA / kind / p)
print(counts)
assert counts['test'] == 3 and counts['val'] == 12, counts

names = re.findall(r'^\s+\d+:\s*(\S+)', (IMAGES.parent / 'data.yaml').read_text(), re.M)
YAML = DATA / 'data.yaml'
YAML.write_text(f'path: {DATA}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n'
                + ''.join(f'  {i}: {n}\n' for i, n in enumerate(names)))
print(YAML.read_text())

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s-seg.pt')
model.train(
    data=str(YAML), imgsz=1280, epochs=150, patience=40, batch=6, seed=0,
    project='/content/runs', name='train', exist_ok=True,
    # the board is seen from many sides, and colour tells the factions apart
    degrees=10, flipud=0.5, fliplr=0.5, scale=0.5, hsv_h=0.01, copy_paste=0.3, close_mosaic=15,
)
RUN = Path(model.trainer.save_dir)
print(RUN)

In [ ]:
import json
from collections import Counter

best = YOLO(RUN / 'weights/best.pt')
report = {}
for split in ('val', 'test'):
    m = best.val(data=str(YAML), split=split, imgsz=1280, project='/content/runs', name=split, exist_ok=True, plots=True)
    report[split] = {
        'seg_mAP50': float(m.seg.map50), 'seg_mAP50_95': float(m.seg.map),
        'seg_AP50_per_class': {names[int(c)]: float(a) for c, a in zip(m.ap_class_index, m.seg.ap50)},
    }

# what the bot needs: how many pieces of each class are on the frame
frames = {}
preds = best.predict(sorted(str(p) for p in (DATA / 'images/test').glob('*.jpg')), imgsz=1280, conf=0.25,
                     save=True, project='/content/runs', name='test_pictures', exist_ok=True, verbose=False)
for r in preds:
    stem = Path(r.path).stem
    true = Counter(int(l.split()[0]) for l in (DATA / 'labels/test' / f'{stem}.txt').read_text().splitlines())
    got = Counter(int(c) for c in r.boxes.cls.tolist())
    frames[stem] = {names[c]: {'true': true[c], 'found': got[c]} for c in range(len(names)) if true[c] or got[c]}
report['test_counts'] = frames
Path('/content/runs/metrics.json').write_text(json.dumps(report, indent=1))
print(json.dumps(report, indent=1))

In [ ]:
!cp /content/runs/train/weights/best.pt /content/runs/piece_seg_v1.pt
!cd /content/runs && zip -qr /content/rootbot_model.zip piece_seg_v1.pt metrics.json train/results.csv train/results.png test test_pictures -x '*.cache'
from google.colab import files
files.download('/content/rootbot_model.zip')